# Overground Spatiotemporal Gait Analysis (from Vicon Events)

This notebook computes spatiotemporal gait parameters for overground walking trials, using gait events that have already been identified within Vicon Nexus and exported alongside the marker trajectories. Compared to my earlier treadmill notebook (which detects events from scratch using marker velocity sign changes), this analysis trusts Vicon's event labels and focuses on parameter computation and clinical interpretation.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived. Reflects my analytical approach during PhD dissertation research.

**Differences from the treadmill notebook**:
- Walking mode is overground, not treadmill (no belt-speed correction)
- Gait events come from Vicon export, not custom detection
- Adds normalized stride and step speeds (cm/s)
- Adds single-support and double-support percentages
- Includes a sanity-check visualization section

---


## 1. Setup and Data Import

Load packages and read the trial. Vicon Nexus exports use a semicolon-delimited CSV with a specific multi-section layout (Joints, Model Outputs, Trajectories), so I keep the delimiter explicit and split into rows for parsing in the next section.

The naming convention is `{Subject} Trial {N}.csv` (e.g., `S01 Trial 2.csv`).

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import os,sys
import seaborn as sns

In [ ]:
# Check current working directory
os.getcwd()

In [ ]:
# Change directory if necessary
#newdirectory = input("Path: ")
#os.chdir(newdirectory)

# Check the changed Directoty
os.getcwd()


In [ ]:
# File import by filename
Subject = input("Subject: ")
Trial = input("Trial: ")
filename = Subject + " "+ "Trial" + " " + Trial

df = pd.read_csv(filename + '.csv', delimiter=';')

df.head()

Split the `Events` column on commas to expose the section headers and per-event fields. This makes the section structure searchable.

In [ ]:
df_split = df['Events'].str.split(',', expand=True)

## 2. Identify Vicon CSV Sections

A Vicon Nexus CSV export contains several stacked sections (Joints, Model Outputs, Trajectories), each with its own header row and data block. Rather than hard-code row offsets (which break whenever the export configuration changes), I scan the first column for the section labels and record their row indices. Downstream slicing then refers to these landmarks.

In [ ]:
for idx in df.index:
     if df_split.iloc[idx, 0] == "Joints" :
         Joint_idx = idx
     if df_split.iloc[idx, 0] == "Model Outputs" :
         MO_idx = idx
     if df_split.iloc[idx, 0] == "Trajectories" :
         TRJ_idx = idx
         

Joint_idx, MO_idx, TRJ_idx

## 3. Extract Gait Events

The events block sits at the top of the CSV (above the Joints section). Each row records a gait event with `Subject`, `Context` (Left or Right), `Name` (Foot Strike or Foot Off), and `Time (s)`.

In [ ]:
# Create Gait Event Dataframe
Gevent_df = df_split.iloc[2:Joint_idx, :4]
Gevent_df.columns = df_split.iloc[1,:4]
Gevent_df['Time (s)']= Gevent_df['Time (s)'].astype(float)

Gevent_df

### Convert event times to frame indices

The marker trajectories are sampled at 100 Hz, so I convert event times (seconds) to integer frame indices by multiplying by 100. This lets me look up the exact marker positions at each event later.

In [ ]:
# Gait Event Dataframe data type change

Gevent_df['idx']= Gevent_df['Time (s)']*100
Gevent_df['idx']= Gevent_df['idx'].astype(int)
Gevent_df

## 4. Extract Marker Trajectories

Locate the heel and toe marker columns within the Trajectories section. The column names follow Vicon's `{Subject}:{Marker}` convention (e.g., `S01:LHEE`). Each marker spans three consecutive columns for X, Y, Z.

In [ ]:
# Get index for Trajectories data

TRJ = df_split.iloc[TRJ_idx + 3:, :]
TRJ.columns = df_split.iloc[TRJ_idx + 2, :]

LHEE_index = TRJ.columns.get_loc(Subject + ":LHEE")
LTOE_index = TRJ.columns.get_loc(Subject + ":LTOE")

RHEE_index = TRJ.columns.get_loc(Subject + ":RHEE")
RTOE_index = TRJ.columns.get_loc(Subject + ":RTOE")

LHEE_index, LTOE_index, RHEE_index, RTOE_index

Build per-foot trajectory dataframes covering the heel and toe markers from the start of the trial.

In [ ]:
# Create Trajectories Dataframe

TRJ_df_L = TRJ.iloc[:, LHEE_index : LTOE_index + 3].reset_index(drop=True)
TRJ_df_R = TRJ.iloc[:, RHEE_index : RTOE_index + 3].reset_index(drop=True)
TRJ_df_FRAME = TRJ.iloc[:, 0].reset_index(drop=True)

TRJ_df = pd.concat([TRJ_df_FRAME, TRJ_df_L, TRJ_df_R], axis=1)
TRJ_df.columns = ['FRAME', 'LHEEX', 'LHEEY', 'LHEEZ', 'LTOEX', 'LTOEY', 'LTOEZ', 'RHEEX', 'RHEEY', 'RHEEZ', 'RTOEX', 'RTOEY', 'RTOEZ']
TRJ_df = TRJ_df.drop([0, 1])

TRJ_df = TRJ_df.astype(float)
TRJ_df['FRAME'] = TRJ_df['FRAME'].astype(int)

# Trajectory Dataframe index setting as 'FRAME'

TRJ_df = TRJ_df.set_index('FRAME')

TRJ_df.head()

## 5. Build Bilateral HS/TO Event DataFrame

Reorganize the long-format Vicon events into a wide-format dataframe with one row per gait cycle and four columns: left heel strike (`LHS`), left toe off (`LTO`), right heel strike (`RHS`), right toe off (`RTO`). All values are frame indices, which makes downstream interval and trajectory calculations straightforward.

In [ ]:
# Make datafrma as Heelstrike and Toeoff

dict1 ={}
dict2 ={}
dict3 ={}
dict4 ={}

LHS = []
LTO = []
RHS = []
RTO = []

for idx in Gevent_df.index:
    if Gevent_df.loc[idx, 'Context'] == "Left" :
        if Gevent_df.loc[idx, 'Name'] == "Foot Strike" :
            LHSidx = Gevent_df.loc[idx, "idx"]
            LHS.append(LHSidx)
        else :
            LTOidx = Gevent_df.loc[idx, "idx"]
            LTO.append(LTOidx)
    else :
        if Gevent_df.loc[idx, 'Name'] == "Foot Strike" :
            RHSidx = Gevent_df.loc[idx, "idx"]
            RHS.append(RHSidx)
        else :
            RTOidx = Gevent_df.loc[idx, "idx"]
            RTO.append(RTOidx)
        
dict1['LHS'] = LHS
dict2['LTO'] = LTO
dict3['RHS'] = RHS
dict4['RTO'] = RTO

LHS_df = pd.DataFrame(dict1)
LTO_df = pd.DataFrame(dict2)
RHS_df = pd.DataFrame(dict3)
RTO_df = pd.DataFrame(dict4)

HSTO = pd.concat([LHS_df, LTO_df, RHS_df, RTO_df], axis=1)
HSTO = HSTO.astype('Int64')
HSTO = HSTO.dropna()
HSTO

## 6. Attach Marker Positions at Events

For each HS event, pull the corresponding heel marker X and Y positions from the trajectory dataframe. These are needed for step length (Y direction) and step width (X direction) calculations downstream.

In [ ]:
# Get trajectory data for gait event

df1 = HSTO
df2 = TRJ_df

LHEEX = []
LHEEY = []
RHEEX = []
RHEEY = []

for idx in df1.index:
    idxl = df1.loc[idx, 'LHS']
    LHX = df2.loc[idxl, 'LHEEX']
    LHEEX.append(LHX)
    LHY = df2.loc[idxl, 'LHEEY']
    LHEEY.append(LHY)
    idxr = df1.loc[idx, 'RHS']
    RHX = df2.loc[idxr, 'RHEEX']
    RHEEX.append(RHX)
    RHY = df2.loc[idxr, 'RHEEY']
    RHEEY.append(RHY)

df1['LHEEX'] = LHEEX
df1['LHEEY'] = LHEEY
df1['RHEEX'] = RHEEX
df1['RHEEY'] = RHEEY

df1

In [ ]:
# Copy dataframe for satiotemporal calculation

strike_df = df1.copy()
strike_df

## 7. Spatiotemporal Parameters

### Stride time and length

Stride spans two consecutive ipsilateral heel-strike events. Because the walking is overground, no belt-speed correction is needed — the heel position difference directly gives stride length.

In [ ]:
# Calculate stride time

Rstridetime = []
Lstridetime = []

for idx in strike_df.index:
    if idx == 0:
        Rstridetime.append(0)
        Lstridetime.append(0)
    else:
        Rstride_time = strike_df.loc[idx, 'RHS'] - strike_df.loc[idx - 1, 'RHS']
        Rstridetime.append(Rstride_time)
        Lstride_time = strike_df.loc[idx, 'LHS'] - strike_df.loc[idx - 1 , 'LHS']
        Lstridetime.append(Lstride_time)
    
        

strike_df['Rstride_time'] = Rstridetime
strike_df['Lstride_time'] = Lstridetime

strike_df

In [ ]:
# Calculate stride Length

RstrideLength = []
LstrideLength = []

for idx in strike_df.index:
    if idx == 0:
        RstrideLength.append(0)
        LstrideLength.append(0)
    else:
        Rstride_Length = abs(strike_df.loc[idx, 'RHEEX'] - strike_df.loc[idx - 1, 'RHEEX'])
        RstrideLength.append(Rstride_Length)
        Lstride_Length = abs(strike_df.loc[idx, 'LHEEX'] - strike_df.loc[idx - 1 , 'LHEEX'])
        LstrideLength.append(Lstride_Length)
    
        

strike_df['Rstride_length'] = RstrideLength
strike_df['Lstride_length'] = LstrideLength

strike_df

### Normalized stride speed

Stride length is converted from mm to cm by dividing by 10, then divided by stride time to give cm/s. This is the standard clinical unit for gait speed reporting.

In [ ]:
# Calculate normalized Stride Speed (unit in cm/s)

strike_df['Rstride_speed'] = ((strike_df['Rstride_length']/10)/strike_df['Rstride_time'])
strike_df['Lstride_speed'] = ((strike_df['Lstride_length']/10)/strike_df['Lstride_time'])
strike_df

### Stance and swing time

Stance spans HS to ipsilateral TO. Swing spans TO to the next ipsilateral HS. The branching handles whichever event came first in the trial, since the alternation pattern propagates from there.

In [ ]:
# Calculate Stance time


Rstancetime = []
Lstancetime = []

for idx in strike_df.index:
    if strike_df.loc[idx, 'LHS'] < strike_df.loc[idx, 'LTO']:
        Lstance_time = strike_df.loc[idx, 'LTO'] - strike_df.loc[idx, 'LHS']
        Lstancetime.append(Lstance_time)
        
    else:
        if idx == 0:
            Lstancetime.append(0)
        else:
            Lstance_time = strike_df.loc[idx, 'LTO'] - strike_df.loc[idx - 1, 'LHS']
            Lstancetime.append(Lstance_time)

for idx in strike_df.index:
    if strike_df.loc[0, 'RHS'] < strike_df.loc[0, 'RTO']:
        Rstance_time = strike_df.loc[idx, 'RTO'] - strike_df.loc[idx, 'RHS']
        Rstancetime.append(Rstance_time)
        
    else:
        if idx == 0:
            Rstancetime.append(0)
        else:
            Rstance_time = strike_df.loc[idx, 'RTO'] - strike_df.loc[idx - 1, 'RHS']
            Rstancetime.append(Rstance_time)
            

strike_df['Rstance_time'] = Rstancetime
strike_df['Lstance_time'] = Lstancetime

strike_df

In [ ]:
# Calculate Swing time


Rswingtime = []
Lswingtime = []

for idx in strike_df.index:
    if strike_df.loc[idx, 'LHS'] < strike_df.loc[idx, 'LTO']:
        if idx == 0:
            Lswingtime.append(0)
        else:
            Lswing_time = strike_df.loc[idx, 'LHS'] - strike_df.loc[idx - 1,'LTO']
            Lswingtime.append(Lswing_time)
    else:
        Lswing_time = strike_df.loc[idx, 'LHS'] - strike_df.loc[idx,'LTO']
        Lswingtime.append(Lswing_time)

for idx in strike_df.index:
    if strike_df.loc[0, 'RHS'] < strike_df.loc[0, 'RTO']:
        if idx == 0:
            Rswingtime.append(0)
        else:
            Rswing_time = strike_df.loc[idx, 'RHS'] - strike_df.loc[idx - 1,'RTO']
            Rswingtime.append(Rswing_time)
    else:
        Rswing_time = strike_df.loc[idx, 'RHS'] - strike_df.loc[idx,'RTO']
        Rswingtime.append(Rswing_time)
        
        

strike_df['Rswing_time'] = Rswingtime
strike_df['Lswing_time'] = Lswingtime

strike_df

### Stance and swing as a percentage of gait cycle

Standard reporting normalizes stance and swing to percentages of the full gait cycle.

In [ ]:
# Caculate stance and swing in Percentage

Rstance = []
Lstance = []
Rswing = []
Lswing = []


for idx in strike_df.index:
    if strike_df.loc[idx, 'LHS'] < strike_df.loc[idx, 'LTO']:
        if idx == 0:
            Lstance.append(0)
            Lswing.append(0) 
        else:
            Lstance_per = (strike_df.loc[idx - 1, 'Lstance_time']/(strike_df.loc[idx, 'Lswing_time'] + strike_df.loc[idx - 1,'Lstance_time']))*100
            Lstance.append(Lstance_per)
            Lswing_per = (strike_df.loc[idx, 'Lswing_time']/(strike_df.loc[idx, 'Lswing_time'] + strike_df.loc[idx - 1,'Lstance_time']))*100
            Lswing.append(Lswing_per)
    else:
        if idx == 0:
            Lstance.append(0)
            Lswing.append(0) 
        else:
            Lstance_per = (strike_df.loc[idx, 'Lstance_time']/(strike_df.loc[idx - 1, 'Lswing_time'] + strike_df.loc[idx,'Lstance_time']))*100
            Lstance.append(Lstance_per)
            Lswing_per = (strike_df.loc[idx - 1, 'Lswing_time']/(strike_df.loc[idx - 1, 'Lswing_time'] + strike_df.loc[idx,'Lstance_time']))*100
            Lswing.append(Lswing_per)

for idx in strike_df.index:
    if strike_df.loc[0, 'RHS'] < strike_df.loc[0, 'RTO']:
        if idx == 0:
            Rstance.append(0)
            Rswing.append(0) 
        else:
            Rstance_per = (strike_df.loc[idx - 1, 'Rstance_time']/(strike_df.loc[idx, 'Rswing_time'] + strike_df.loc[idx - 1,'Rstance_time']))*100
            Rstance.append(Rstance_per)
            Rswing_per = (strike_df.loc[idx, 'Rswing_time']/(strike_df.loc[idx, 'Rswing_time'] + strike_df.loc[idx - 1,'Rstance_time']))*100
            Rswing.append(Rswing_per)
    else:
        if idx == 0:
            Rstance.append(0)
            Rswing.append(0) 
        else:
            Rstance_per = (strike_df.loc[idx, 'Rstance_time']/(strike_df.loc[idx - 1, 'Rswing_time'] + strike_df.loc[idx,'Rstance_time']))*100
            Rstance.append(Rstance_per)
            Rswing_per = (strike_df.loc[idx - 1, 'Rswing_time']/(strike_df.loc[idx - 1, 'Rswing_time'] + strike_df.loc[idx,'Rstance_time']))*100
            Rswing.append(Rswing_per)
        

strike_df['Rstance%'] = Rstance
strike_df['Lstance%'] = Lstance
strike_df['Rswing%'] = Rswing
strike_df['Lswing%'] = Lswing

strike_df

### Step time

Step spans contralateral to ipsilateral HS. The same first-event-determines-alternation pattern applies.

In [ ]:
# Calculate step time


Rsteptime = []
Lsteptime = []

for idx in strike_df.index:
    if strike_df.loc[idx, 'LHS'] < strike_df.loc[idx, 'RHS']:
        if idx == 0:
            Rstep_time = strike_df.loc[idx, 'RHS'] - strike_df.loc[idx, 'LHS']
            Rsteptime.append(Rstep_time)
            Lsteptime.append(0)
        else:
            Rstep_time = strike_df.loc[idx, 'RHS'] - strike_df.loc[idx, 'LHS']
            Rsteptime.append(Rstep_time)
            Lstep_time = strike_df.loc[idx, 'LHS'] - strike_df.loc[idx - 1,'RHS']
            Lsteptime.append(Lstep_time)
    else:
        if idx ==0:
            Rsteptime.append(0)
            Lstep_time = strike_df.loc[idx, 'LHS'] - strike_df.loc[idx,'RHS']
            Lsteptime.append(Lstep_time)
        else:
            Rstep_time = strike_df.loc[idx, 'RHS'] - strike_df.loc[idx - 1, 'LHS']
            Rsteptime.append(Rstep_time)
            Lstep_time = strike_df.loc[idx, 'LHS'] - strike_df.loc[idx,'RHS']
            Lsteptime.append(Lstep_time)
        

strike_df['Rstep_time'] = Rsteptime
strike_df['Lstep_time'] = Lsteptime

strike_df

### Step length

For overground walking, step length is simply the absolute heel Y-position difference between contralateral and ipsilateral HS — no belt correction needed.

In [ ]:
#Calculate Step Length

Rsteplength = []
Lsteplength = []

for idx in strike_df.index:
    if strike_df.loc[idx, 'LHS'] < strike_df.loc[idx, 'RHS']:
        if idx == 0:
            Rstep_length = abs(strike_df.loc[idx, 'RHEEX'] - strike_df.loc[idx, 'LHEEX'])
            Rsteplength.append(Rstep_length)
            Lsteplength.append(0)
        else:
            Rstep_length = abs(strike_df.loc[idx, 'RHEEX'] - strike_df.loc[idx, 'LHEEX']) 
            Rsteplength.append(Rstep_length)
            Lstep_length = abs(strike_df.loc[idx, 'LHEEX'] - strike_df.loc[idx - 1,'RHEEX'])
            Lsteplength.append(Lstep_length)
    else:
        if idx ==0:
            Rsteplength.append(0)
            Lstep_length = abs(strike_df.loc[idx, 'LHEEX'] - strike_df.loc[idx,'RHEEX'])
            Lsteplength.append(Lstep_length)
        else:
            Rstep_length = abs(strike_df.loc[idx, 'RHEEX'] - strike_df.loc[idx - 1, 'LHEEX'])
            Rsteplength.append(Rstep_length)
            Lstep_length = abs(strike_df.loc[idx, 'LHEEX'] - strike_df.loc[idx,'RHEEX'])
            Lsteplength.append(Lstep_length)
        

strike_df['Rstep_length'] = Rsteplength
strike_df['Lstep_length'] = Lsteplength

strike_df

### Normalized step speed (cm/s)

In [ ]:
# Calculate normalized step speed (unit in cm/s)
strike_df['Rstep_speed'] = (strike_df['Rstep_length']/10)/strike_df['Rstep_time']
strike_df['Lstep_speed'] = (strike_df['Lstep_length']/10)/strike_df['Lstep_time']

strike_df

### Step width

Lateral (X-direction) distance between heel positions at contralateral HS events.

In [ ]:
#Calculate Step Width

Rstepwidth = []
Lstepwidth = []

for idx in strike_df.index:
    if strike_df.loc[idx, 'LHS'] < strike_df.loc[idx, 'RHS']:
        if idx == 0:
            Rstep_width = abs(strike_df.loc[idx, 'RHEEY'] - (strike_df.loc[idx, 'LHEEY']))
            Rstepwidth.append(Rstep_width)
            Lstepwidth.append(0)
        else:
            Rstep_width = abs(strike_df.loc[idx, 'RHEEY'] - (strike_df.loc[idx, 'LHEEY']))
            Rstepwidth.append(Rstep_width)
            Lstep_width = abs(strike_df.loc[idx, 'LHEEY'] - (strike_df.loc[idx - 1,'RHEEY']))
            Lstepwidth.append(Lstep_width)
    else:
        if idx ==0:
            Rstepwidth.append(0)
            Lstep_width = abs(strike_df.loc[idx, 'LHEEY'] - (strike_df.loc[idx,'RHEEY']))
            Lstepwidth.append(Lstep_width)
        else:
            Rstep_width = abs(strike_df.loc[idx, 'RHEEY'] - (strike_df.loc[idx - 1, 'LHEEY']))
            Rstepwidth.append(Rstep_width)
            Lstep_width = abs(strike_df.loc[idx, 'LHEEY'] - (strike_df.loc[idx,'RHEEY']))
            Lstepwidth.append(Lstep_width)
        

strike_df['Rstep_width'] = Rstepwidth
strike_df['Lstep_width'] = Lstepwidth

strike_df

### Single and double support percentages

These are clinically meaningful metrics that the treadmill notebook did not include:

- **Single support** of one limb = the contralateral limb's swing duration relative to its step time
- **Double support** is the residual percentage when both limbs are in contact with the ground

These two metrics together describe how walking is distributed between single-leg and bilateral support phases — useful for analyzing gait stability and asymmetry, particularly in clinical populations.

In [ ]:
# Calculate single and double support percentage

strike_df['Rsingle'] = (strike_df['Lswing_time']/strike_df['Lstep_time'])*100
strike_df['Lsingle'] = (strike_df['Rswing_time']/strike_df['Rstep_time'])*100


strike_df['Rdouble'] = 100 - (strike_df['Lswing_time']/strike_df['Lstep_time'])*100
strike_df['Ldouble'] = 100 - (strike_df['Rswing_time']/strike_df['Rstep_time'])*100


strike_df

## 8. Output

Save per-cycle values to a trial CSV, and also append to a per-subject aggregate CSV for trial-to-trial comparison within a subject.

In [ ]:
# Save spatiotemporal data in csv file (by trial)

strike_df.to_csv(filename + '_data.csv',  index=False)

In [ ]:
# Save spatiotemporal data in csv (by subject)

if not os.path.exists(Subject + '_data.csv'):
    strike_df.to_csv(Subject + '_data.csv', index=filename, mode='w')
else:
    strike_df.to_csv(Subject + '_data.csv', index=filename, mode='a')

## 9. Visualization for Sanity Check

Before trusting the calculated parameters, I plot the detected events and corresponding marker positions to verify nothing is anomalous. This catches several common problems:

- **Missed or extra events** show up as gaps or doublets in the event-time scatter
- **Coordinate frame errors** (e.g., axis flipped on a particular trial) become obvious in the XY trajectory plot
- **Trial-level outliers** (e.g., a subject who took a sudden long step) stand out as visual outliers in any of the plots

I treat these plots as a routine sanity step rather than a one-time check. Looking at the data, not just the summary numbers, is what makes me confident in the per-trial values that go into the aggregate CSV.

**Event timing across the trial.** Each scatter point is one gait event. Confirms events are roughly evenly spaced and that there is no missing block.

In [ ]:
sns.scatterplot(data=df1.iloc[:,:4])

**Heel marker X-position at left vs right heel strikes.** Confirms left and right heels are on opposite sides (different X coordinates) at HS, as anatomy requires.

In [ ]:
sns.scatterplot(data=df1, x='LHS',y='LHEEX'), sns.scatterplot(data=df1, x='RHS',y='RHEEX')

**XY trajectory of the heel markers at heel strikes.** Should show a clean walking pattern — heel positions progressing forward (Y) while alternating laterally (X).

In [ ]:
sns.scatterplot(data=df1, x='LHEEX',y='LHEEY'), sns.scatterplot(data=df1, x='RHEEX',y='RHEEY')

**Events sorted by frame index.** Useful for spotting any out-of-order events that the previous plots might miss.

In [ ]:
Gevent_df.sort_values(by=['idx'])